# T17 — Fidelity Comparison & Data Masking

Compare real vs synthetic data quality and mask PII in real datasets.

**What you'll learn:**
- Score synthetic data fidelity against real data
- Read markdown fidelity reports
- Mask PII columns automatically
- Configure masking exclusions and overrides

## 1. Generate Real + Synthetic Data

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

real_data = {
    "customer": pd.DataFrame({
        "customer_id": range(1, 1001),
        "name": [f"Person {i}" for i in range(1, 1001)],
        "email": [f"user{i}@company.com" for i in range(1, 1001)],
        "phone": [f"555-{i:04d}" for i in range(1, 1001)],
        "age": rng.normal(40, 12, 1000).astype(int).clip(18, 90),
        "tier": rng.choice(["basic", "silver", "gold"], 1000, p=[0.6, 0.3, 0.1]),
        "spend": rng.lognormal(6, 1.0, 1000).round(2),
    }),
}

# Slightly different synthetic version
rng2 = np.random.default_rng(99)
synth_data = {
    "customer": pd.DataFrame({
        "customer_id": range(1, 1001),
        "name": [f"Synth {i}" for i in range(1, 1001)],
        "email": [f"synth{i}@fake.com" for i in range(1, 1001)],
        "phone": [f"555-{i:04d}" for i in range(1, 1001)],
        "age": rng2.normal(42, 14, 1000).astype(int).clip(18, 90),
        "tier": rng2.choice(["basic", "silver", "gold"], 1000, p=[0.55, 0.30, 0.15]),
        "spend": rng2.lognormal(6.2, 1.1, 1000).round(2),
    }),
}
print(f"Real: {len(real_data['customer'])} rows, Synthetic: {len(synth_data['customer'])} rows")

Real: 1000 rows, Synthetic: 1000 rows


## 2. Compare Fidelity

In [2]:
from sqllocks_spindle.inference.comparator import FidelityComparator

comp = FidelityComparator()
report = comp.compare(real_data, synth_data)

print(f"Overall Fidelity Score: {report.overall_score:.1f}/100")
print()
print(report.summary())

Overall Fidelity Score: 77.9/100

Fidelity Report
Overall Score: 77.9/100

  customer — Score: 77.9/100 (1000 real, 1000 synth rows)
    age: 87.9/100 (type_match=True)
    customer_id: 100.0/100 (type_match=True)
    email: 42.9/100 (type_match=True)
    name: 42.9/100 (type_match=True)
    phone: 100.0/100 (type_match=True)
    spend: 80.7/100 (type_match=True)
    tier: 90.8/100 (type_match=True)


## 3. Detailed Column Scores

In [3]:
for table_name, table_fidelity in report.tables.items():
    print(f"\n=== {table_name} (score: {table_fidelity.score:.1f}) ===")
    for col_name, col_fidelity in table_fidelity.columns.items():
        print(f"  {col_name:20s} score={col_fidelity.score:5.1f}  "
              f"dtype_match={col_fidelity.dtype_match}  "
              f"null_delta={col_fidelity.null_rate_delta:.3f}")


=== customer (score: 77.9) ===
  age                  score= 87.9  dtype_match=True  null_delta=0.000
  customer_id          score=100.0  dtype_match=True  null_delta=0.000
  email                score= 42.9  dtype_match=True  null_delta=0.000
  name                 score= 42.9  dtype_match=True  null_delta=0.000
  phone                score=100.0  dtype_match=True  null_delta=0.000
  spend                score= 80.7  dtype_match=True  null_delta=0.000
  tier                 score= 90.8  dtype_match=True  null_delta=0.000


## 4. Markdown Report

In [4]:
md = report.to_markdown()
print(md[:500])  # First 500 chars

# Fidelity Report

**Overall Score: 77.9/100**

## customer — 77.9/100

| Column | Score | Type Match | Null Delta | KS Stat | Chi2 Stat |
|--------|-------|------------|------------|---------|-----------|
| age | 87.9 | ✓ | 0.000 | 0.132 | — |
| customer_id | 100.0 | ✓ | 0.000 | 0.000 | — |
| email | 42.9 | ✓ | 0.000 | — | 9999999999000.0 |
| name | 42.9 | ✓ | 0.000 | — | 9999999999000.0 |
| phone | 100.0 | ✓ | 0.000 | — | 0.0 |
| spend | 80.7 | ✓ | 0.000 | 0.094 | — |
| tier | 90.8 | ✓ | 0.000


## 5. Mask PII in Real Data

In [5]:
from sqllocks_spindle.inference.masker import DataMasker, MaskConfig

masker = DataMasker()
mask_result = masker.mask(
    tables=real_data,
    config=MaskConfig(
        seed=42,
        preserve_nulls=True,
        preserve_fks=True,
        exclude_columns=["customer_id", "tier"],
    ),
)

print(f"Masked {mask_result.columns_masked} columns")
print("\nOriginal:")
print(real_data['customer'][['name', 'email', 'phone', 'age']].head())
print("\nMasked:")
print(mask_result.tables['customer'][['name', 'email', 'phone', 'age']].head())

Masked {'customer': ['name', 'email', 'phone']} columns

Original:
       name              email     phone  age
0  Person 1  user1@company.com  555-0001   43
1  Person 2  user2@company.com  555-0002   27
2  Person 3  user3@company.com  555-0003   49
3  Person 4  user4@company.com  555-0004   51
4  Person 5  user5@company.com  555-0005   18

Masked:
              name                       email              phone  age
0     Allison Hill       qjacobson@example.org         6502166799   43
1      Noah Rhodes          ylopez@example.com  767.389.9730x8069   27
2  Angie Henderson       seanbaker@example.com   567-917-9576x024   49
3    Daniel Wagner        jeremy49@example.org  (873)783-9597x246   51
4  Cristian Santos  karencontreras@example.org  329.834.1722x3297   18


## 6. CLI Equivalent

```bash
# Compare fidelity
spindle compare ./real_data/ ./synthetic/ --format csv --output fidelity.md

# Mask PII
spindle mask ./real_data/ --output ./masked/ --seed 42 --exclude customer_id
```